In [4]:
import pandas as pd

# ----------------------------
# Load CSV files
# ----------------------------
activity = pd.read_csv("dailyActivity_merged1.csv")
sleep = pd.read_csv("sleepDay_merged.csv")
weight = pd.read_csv("weightLogInfo_merged.csv")

# ----------------------------
# Standardize date columns
# ----------------------------
activity["Date"] = pd.to_datetime(activity["ActivityDate"])
sleep["Date"] = pd.to_datetime(sleep["SleepDay"])
weight["Date"] = pd.to_datetime(weight["Date"])

# ----------------------------
# Drop old date columns
# ----------------------------
activity.drop(columns=["ActivityDate"], inplace=True)
sleep.drop(columns=["SleepDay"], inplace=True)

# ----------------------------
# Aggregate sleep (multiple records per day possible)
# ----------------------------
sleep_agg = sleep.groupby(["Id", "Date"], as_index=False).mean(numeric_only=True)

# ----------------------------
# Merge datasets
# ----------------------------
merged = pd.merge(activity, sleep_agg, on=["Id", "Date"], how="left")
merged = pd.merge(merged, weight, on=["Id", "Date"], how="left")

# ----------------------------
# Save final dataset
# ----------------------------
merged.to_csv("fitbit_merged_final.csv", index=False)

print("✅ All datasets merged successfully into fitbit_merged_final.csv")


C:\Users\rohit\AppData\Local\Temp\ipykernel_20888\2133898936.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sleep["Date"] = pd.to_datetime(sleep["SleepDay"])
C:\Users\rohit\AppData\Local\Temp\ipykernel_20888\2133898936.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  weight["Date"] = pd.to_datetime(weight["Date"])


PermissionError: [Errno 13] Permission denied: 'fitbit_merged_final.csv'

In [6]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

# ----------------------------
# Load merged dataset
# ----------------------------
df = pd.read_csv("fitbit_merged_final.csv")

# ----------------------------
# Fix date parsing (explicit format)
# ----------------------------
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

df["day"] = df["Date"].dt.day
df["month"] = df["Date"].dt.month
df["weekday"] = df["Date"].dt.weekday

df.drop(columns=["Date"], inplace=True)

# ----------------------------
# Drop ID from features
# ----------------------------
ids = df["Id"]
df.drop(columns=["Id"], inplace=True)

# ----------------------------
# DROP columns with 100% null values
# ----------------------------
df = df.dropna(axis=1, how="all")

print("Remaining columns:", df.columns.tolist())

# ----------------------------
# Impute missing values (median)
# ----------------------------
imputer = SimpleImputer(strategy="median")
df_imputed = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)

# ----------------------------
# Scale with RobustScaler
# ----------------------------
scaler = RobustScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_imputed),
    columns=df.columns
)

# ----------------------------
# Reattach ID
# ----------------------------
df_scaled.insert(0, "Id", ids.values)

# ----------------------------
# Save final clean dataset
# ----------------------------
df_scaled.to_csv("fitbit_preprocessed.csv", index=False)

print("✅ Preprocessing complete. Saved as fitbit_preprocessed.csv")


Remaining columns: ['TotalSteps', 'TotalDistance', 'TrackerDistance', 'LoggedActivitiesDistance', 'VeryActiveDistance', 'ModeratelyActiveDistance', 'LightActiveDistance', 'SedentaryActiveDistance', 'VeryActiveMinutes', 'FairlyActiveMinutes', 'LightlyActiveMinutes', 'SedentaryMinutes', 'Calories', 'TotalSleepRecords', 'TotalMinutesAsleep', 'TotalTimeInBed', 'day', 'month', 'weekday']
✅ Preprocessing complete. Saved as fitbit_preprocessed.csv


In [7]:
df = pd.read_csv("fitbit_preprocessed.csv")
print("Total NaNs:", df.isnull().sum().sum())
print("Shape:", df.shape)


Total NaNs: 0
Shape: (940, 20)


In [8]:
EXCLUDED = ["Id"]


In [9]:
import pandas as pd
from sklearn.ensemble import IsolationForest
import joblib

# ----------------------------
# Load preprocessed dataset
# ----------------------------
df = pd.read_csv("fitbit_preprocessed.csv")

# ----------------------------
# Separate features
# ----------------------------
X = df.drop(columns=["Id"])

# ----------------------------
# Train Isolation Forest
# ----------------------------
model = IsolationForest(
    n_estimators=400,
    contamination=0.05,   # top 5% anomalies
    random_state=42,
    n_jobs=-1
)

model.fit(X)

# ----------------------------
# Save trained model
# ----------------------------
joblib.dump(model, "fitbit_anomaly_model.pkl")

print("✅ Fitbit anomaly detection model trained and saved.")


✅ Fitbit anomaly detection model trained and saved.


In [10]:
import pandas as pd
import joblib

# Load assets
model = joblib.load("fitbit_anomaly_model.pkl")

# Load data
df = pd.read_csv("fitbit_preprocessed.csv")

X = df.drop(columns=["Id"])

# Predict
df["anomaly_flag"] = model.predict(X)          # -1 = anomaly, 1 = normal
df["anomaly_score"] = model.decision_function(X)

# Human-readable label
df["status"] = df["anomaly_flag"].map({
    -1: "Anomalous",
     1: "Normal"
})

# Save results
df.to_csv("fitbit_with_anomalies.csv", index=False)

print("✅ Anomaly detection complete. Results saved.")


✅ Anomaly detection complete. Results saved.
